# Spike A — F5-TTS & ZipVoice → ONNX export

**Goal:** verify we can get a flow-matching TTS checkpoint into a single ONNX
file that ONNX Runtime can load on both CPU (this notebook) and eventually
WebGPU (Spike B).

## Why two models in one notebook

Spike E flagged **ZipVoice-Distill** (123M params, 4 NFE, Apache-2.0, built-in
ONNX export) as likely a better base than F5-TTS. We're going to attempt
both exports here so that after this notebook we have a concrete comparison:
- Does F5's DiT-with-rotary export cleanly? Custom ops, parity, file size.
- Does ZipVoice's Zipformer export cleanly via its first-party tooling?

If only one succeeds, we know which model to build Phase 1 around.
If both succeed, we fall through to Spike B and benchmark both in-browser.

## Runtime

- Colab (free T4) or Kaggle T4 is fine. No A100 required for export itself.
- Expect ~60 min wall clock (most of it is model downloads).

## Artifacts produced

- `f5_teacher.onnx` (~670 MB FP16) — the F5-TTS teacher
- `zipvoice_distill.onnx` (~250 MB FP16) — ZipVoice-Distill
- `parity_report.json` — PyTorch-vs-ORT-CPU RMS error per prompt per model

## 0. Environment

In [ ]:
import sys, subprocess, platform, json, os
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except ImportError:
    print('Torch not yet installed')

# Scratch dir for artifacts
ART_DIR = '/content/spike_a_artifacts'
os.makedirs(ART_DIR, exist_ok=True)
print('Artifacts →', ART_DIR)

In [ ]:
# Install deps. Pin versions to what we know works on T4 / Apple Silicon.
# onnx==1.16 + onnxruntime==1.19 is the baseline both ORT-Web and this
# notebook should agree on.
!pip install -q torch==2.4.0 torchaudio==2.4.0
!pip install -q onnx==1.16.2 onnxruntime==1.19.2 onnxconverter-common==1.14.0
!pip install -q f5-tts==1.0.3 || pip install -q git+https://github.com/SWivid/F5-TTS.git
!pip install -q zipvoice || pip install -q git+https://github.com/k2-fsa/ZipVoice.git
print('Deps installed.')

## 1. Define parity prompts

Three prompts of increasing complexity. We'll run each through PyTorch and
ORT CPU and require RMS(PT, ORT) < 0.05 on the raw model output (mel, not
waveform — we're only testing the generator's export fidelity here, not
the vocoder).

In [ ]:
PARITY_PROMPTS = [
    'Hello world.',
    'The quick brown fox jumps over the lazy dog.',
    'She sells seashells by the seashore; the shells she sells are surely seashells.',
]
RMS_THRESHOLD = 0.05

def rms_diff(a, b):
    import numpy as np
    a = a.astype('float32').flatten()
    b = b.astype('float32').flatten()
    n = min(len(a), len(b))
    return float(np.sqrt(np.mean((a[:n] - b[:n]) ** 2)))

## 2. Path A — F5-TTS teacher export

F5's generator is a Diffusion Transformer. The tricky ops for ONNX export are:
- Rotary position embedding (needs opset 18+)
- `scaled_dot_product_attention` (opset 14+, but watch the mask shape)
- Conditional flow sampling loop — we **do not** export the loop itself, just
  one forward pass. The sampler will live on the client (Reader) side.

We export the single-step model: `(noise_t, t, text_cond, ref_mel) → velocity`.
The client will call this 32 times (or 7 with EPSS, or 4 after distillation).

In [ ]:
import torch
import torch.nn as nn

# Load stock F5 teacher. The exact import path may drift; update per
# SWivid/F5-TTS README when this runs.
try:
    from f5_tts.model import DiT, CFM
    from f5_tts.infer.utils_infer import load_model
    F5_AVAILABLE = True
except ImportError as e:
    print('F5 import failed:', e)
    F5_AVAILABLE = False

In [ ]:
if F5_AVAILABLE:
    # Download the pretrained F5-TTS v1 base checkpoint.
    # (HuggingFace: SWivid/F5-TTS, model_1200000.pt)
    import huggingface_hub as hfh
    ckpt_path = hfh.hf_hub_download('SWivid/F5-TTS', 'F5TTS_v1_Base/model_1250000.safetensors')
    vocab_path = hfh.hf_hub_download('SWivid/F5-TTS', 'F5TTS_v1_Base/vocab.txt')
    print('Checkpoint:', ckpt_path)
    print('Vocab:', vocab_path)

In [ ]:
if F5_AVAILABLE:
    # Wrap the generator in a module with fixed I/O signatures so
    # torch.onnx.export has something deterministic to trace.
    class F5Generator(nn.Module):
        def __init__(self, cfm):
            super().__init__()
            self.cfm = cfm  # the transformer inside CFM, not the sampling loop
        def forward(self, x, t, text, ref_mel, ref_mel_len):
            # x: [B, T, D_mel], t: [B], text: [B, T_text], ref_mel: [B, T_ref, D_mel]
            return self.cfm.transformer(x=x, t=t, text=text, ref_mel=ref_mel,
                                         ref_mel_len=ref_mel_len)

    # TODO(spike): the exact F5 CFM constructor signature + load_model() call
    # needs to be matched to the current SWivid/F5-TTS API. The snippet below
    # is illustrative — the notebook author should mirror F5's
    # `infer_cli.py` preprocessing path so the export mirrors real inference.
    print('TODO: instantiate CFM with the downloaded checkpoint and wrap it.')
    print('Copy the load path from f5-tts/src/f5_tts/infer/infer_cli.py.')

In [ ]:
if F5_AVAILABLE:
    # Dummy inputs for tracing (batch=1, seq=200 mel frames, mel_dim=100,
    # text=100 tokens). Dynamic axes below make seq/text lengths flexible.
    B = 1
    T_MEL = 200
    T_REF = 100
    D_MEL = 100
    T_TEXT = 100

    dummy_x = torch.randn(B, T_MEL, D_MEL)
    dummy_t = torch.rand(B)
    dummy_text = torch.randint(0, 256, (B, T_TEXT))
    dummy_ref = torch.randn(B, T_REF, D_MEL)
    dummy_ref_len = torch.tensor([T_REF])

    f5_onnx = f'{ART_DIR}/f5_teacher.onnx'

    # model = F5Generator(cfm).eval()  # Uncomment once CFM is loaded
    # torch.onnx.export(
    #     model,
    #     (dummy_x, dummy_t, dummy_text, dummy_ref, dummy_ref_len),
    #     f5_onnx,
    #     input_names=['x', 't', 'text', 'ref_mel', 'ref_mel_len'],
    #     output_names=['velocity'],
    #     dynamic_axes={
    #         'x': {1: 'mel_len'},
    #         'text': {1: 'text_len'},
    #         'ref_mel': {1: 'ref_len'},
    #         'velocity': {1: 'mel_len'},
    #     },
    #     opset_version=18,  # required for rotary
    # )
    # print('Exported →', f5_onnx, os.path.getsize(f5_onnx) // 1024 // 1024, 'MB')

In [ ]:
if F5_AVAILABLE and os.path.exists(f'{ART_DIR}/f5_teacher.onnx'):
    # Convert FP32 → FP16 to hit the target download size.
    from onnxconverter_common import float16
    import onnx
    m = onnx.load(f'{ART_DIR}/f5_teacher.onnx')
    m_fp16 = float16.convert_float_to_float16(m, keep_io_types=True)
    onnx.save(m_fp16, f'{ART_DIR}/f5_teacher_fp16.onnx')
    size_mb = os.path.getsize(f'{ART_DIR}/f5_teacher_fp16.onnx') // 1024 // 1024
    print(f'FP16 size: {size_mb} MB  (target < 1500 MB)')

## 3. Path B — ZipVoice-Distill export (using k2-fsa's built-in tooling)

ZipVoice ships its own ONNX exporter. This should Just Work, and if it does
we're largely done — ZipVoice-Distill can replace the whole F5 teacher +
custom-distillation pipeline we were planning.

In [ ]:
# ZipVoice-Distill export via first-party tooling.
# Their command-line entry is `zipvoice.bin.infer_zipvoice_onnx` (inference),
# but the export path is typically at `zipvoice.bin.export_onnx` — check
# their current README when running this; interface has drifted in 2025.
try:
    !python -m zipvoice.bin.export_onnx \
        --model-name zipvoice_distill \
        --output-dir {ART_DIR}/zipvoice \
        --fp16 true
    zv_onnx = f'{ART_DIR}/zipvoice/model.onnx'
    if os.path.exists(zv_onnx):
        size_mb = os.path.getsize(zv_onnx) // 1024 // 1024
        print(f'ZipVoice-Distill ONNX: {size_mb} MB')
    else:
        print('WARN: ZipVoice export did not produce the expected path.')
        print('Check their current docs for the export CLI name.')
except Exception as e:
    print('ZipVoice export failed:', e)
    print('If this fails, fall back to F5 path only.')

## 4. Parity check — PyTorch vs ORT CPU

For each parity prompt, run both the PyTorch original and the exported ONNX
(via `onnxruntime` CPU EP) and compare outputs. We accept RMS < 0.05 on
the raw velocity output. If it's larger, the export is lossy and WebGPU
performance won't save us.

In [ ]:
import numpy as np
import onnxruntime as ort

parity = {'f5': [], 'zipvoice': []}

def run_parity(onnx_path, pytorch_fn, prompts, model_key):
    if not os.path.exists(onnx_path):
        print(f'Skipping {model_key}: no ONNX artifact at {onnx_path}')
        return
    sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    print(f'ORT session loaded for {model_key}. Inputs:',
          [i.name for i in sess.get_inputs()])
    for p in prompts:
        # TODO: build the proper input tensors for each model family.
        # For F5 we need (x, t, text_tokens, ref_mel, ref_mel_len).
        # For ZipVoice we need their tokenized inputs — see their infer script.
        # The parity check should use IDENTICAL inputs through both paths.
        print(f'  [{model_key}] prompt={p!r}  → (populate inputs + run)')
        # pt_out = pytorch_fn(p)
        # ort_out = sess.run(None, build_inputs(p))[0]
        # err = rms_diff(pt_out, ort_out)
        # parity[model_key].append({'prompt': p, 'rms': err, 'pass': err < RMS_THRESHOLD})

run_parity(f'{ART_DIR}/f5_teacher_fp16.onnx', None, PARITY_PROMPTS, 'f5')
run_parity(f'{ART_DIR}/zipvoice/model.onnx', None, PARITY_PROMPTS, 'zipvoice')

## 5. Write parity report

Small JSON that goes into `spikes/results/` — safe to commit (no weights).

In [ ]:
import json, datetime, hashlib

def sha256(path):
    if not os.path.exists(path):
        return None
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(2**20), b''):
            h.update(chunk)
    return h.hexdigest()

report = {
    'spike': 'A',
    'generated_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'environment': {
        'python': sys.version.split()[0],
        'platform': platform.platform(),
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        'torch': torch.__version__,
    },
    'rms_threshold': RMS_THRESHOLD,
    'artifacts': {
        'f5_teacher_fp16': {
            'path': f'{ART_DIR}/f5_teacher_fp16.onnx',
            'size_mb': (os.path.getsize(f'{ART_DIR}/f5_teacher_fp16.onnx') // 1024 // 1024)
                if os.path.exists(f'{ART_DIR}/f5_teacher_fp16.onnx') else None,
            'sha256': sha256(f'{ART_DIR}/f5_teacher_fp16.onnx'),
        },
        'zipvoice_distill': {
            'path': f'{ART_DIR}/zipvoice/model.onnx',
            'size_mb': (os.path.getsize(f'{ART_DIR}/zipvoice/model.onnx') // 1024 // 1024)
                if os.path.exists(f'{ART_DIR}/zipvoice/model.onnx') else None,
            'sha256': sha256(f'{ART_DIR}/zipvoice/model.onnx'),
        },
    },
    'parity': parity,
}

report_path = f'{ART_DIR}/parity_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print('Wrote', report_path)
print(json.dumps(report, indent=2))

## 6. Download artifacts

Pull `parity_report.json` + both ONNX files to your Mac. The JSON goes into
`spikes/results/` in git; the ONNX files go into `~/voice-spike-artifacts/`
(too big for git).

In [ ]:
# Colab-only shim. On Kaggle, use the dataset-output UI.
try:
    from google.colab import files
    files.download(report_path)
    if os.path.exists(f'{ART_DIR}/f5_teacher_fp16.onnx'):
        files.download(f'{ART_DIR}/f5_teacher_fp16.onnx')
    if os.path.exists(f'{ART_DIR}/zipvoice/model.onnx'):
        files.download(f'{ART_DIR}/zipvoice/model.onnx')
except ImportError:
    print('Not in Colab — download manually from', ART_DIR)

## 7. What to write in REPORT.md

Fill in the Spike A section of `spikes/REPORT.md` with:
- F5 export: ✅/❌, size, parity RMS per prompt, any ops that needed
  custom handling
- ZipVoice export: ✅/❌, size, parity RMS per prompt
- Go/no-go verdict. Expected happy path: ZipVoice ✅ → use it as our base.
  Fallback: F5 ✅ only → continue with original plan, add EPSS client-side.
  Red: both ❌ → pivot to Kokoro-82M (smaller, known clean export).